In [1]:
!pip install unsloth transformers datasets accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import json
from tqdm import tqdm
from unsloth import FastLanguageModel
import torch
from transformers import StoppingCriteria, StoppingCriteriaList

# --- 1. CONFIG ---
TEST_SUITE = "/content/drive/MyDrive/SLM_DATA/test_suite_200_FINAL.jsonl"
MODEL_PATH = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed"
OUTPUT_FILE = "/content/drive/MyDrive/SLM_DATA/stage2_FINAL_EVAL_MATCHED.jsonl"

# --- 2. LOAD MODEL ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit = True,
    # fix_mistral_regex = True # Removed, as it's not a valid argument for Qwen2 models
)
FastLanguageModel.for_inference(model)

# --- 3. STOPPING CRITERIA (Prevents Double Vision) ---
class StopOnCodeEnd(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0][-5:])
        return "</code>" in decoded

stop_criteria = StoppingCriteriaList([StopOnCodeEnd()])

# --- 4. RUN MATCHED EVALUATION ---
with open(TEST_SUITE, 'r') as f:
    test_cases = [json.loads(line) for line in f]

print(f"🚀 Starting Final Eval on {len(test_cases)} questions...")

with open(OUTPUT_FILE, 'w') as out_f:
    for case in tqdm(test_cases, desc="🤖 Generating"):
        # We extract the TRUE ID and TRUE PROBLEM from the ground truth
        true_id = case.get("id")
        problem_text = case.get("problem", case.get("instruction", ""))

        # Enhanced Prompting
        prompt = f"""### Instruction:
Solve this competitive programming problem.
1. Use standard I/O (sys.stdin.read() or input()).
2. Output logic must be exactly: <tags>...</tags><think>...</think><code>...</code>.
3. NO example usage or hardcoded variables.

Problem: {problem_text}

### Response:
<tags>"""

        inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens = 1500,
            temperature = 0.1,
            top_p = 0.9,
            stopping_criteria = stop_criteria,
            use_cache = True
        )

        full_response = tokenizer.batch_decode(outputs)[0].split("### Response:\n")[-1].strip()

        # Save with the TRUE ID and INSTRUCTION for the Judge
        result_entry = {
            "id": true_id,
            "instruction": problem_text,
            "prediction": full_response
        }
        out_f.write(json.dumps(result_entry) + "\n")

print(f"\n✅ Evaluation complete! Results saved to: {OUTPUT_FILE}")

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed as a legacy tokenizer.


🚀 Starting Final Eval on 200 questions...


🤖 Generating: 100%|██████████| 200/200 [1:10:59<00:00, 21.30s/it]


✅ Evaluation complete! Results saved to: /content/drive/MyDrive/SLM_DATA/stage2_FINAL_EVAL_MATCHED.jsonl


In [4]:
import json
import subprocess
import sys
import tempfile
import os
import re
from tqdm.auto import tqdm

# --- CONFIGURATION ---
# Point to your NEW matched results and the ORIGINAL test suite
MODEL_SOLUTIONS_FILE = "/content/drive/MyDrive/SLM_DATA/stage2_FINAL_EVAL_MATCHED.jsonl"
GROUND_TRUTH_FILE = "/content/drive/MyDrive/SLM_DATA/test_suite_200_FINAL.jsonl"

def parse_stage2_response(full_text):
    """
    Parses the <tags><think><code> structure.
    Handles potential instruction echoing and whitespace.
    """
    # Isolate the generation after the instruction prompt
    actual_gen = full_text.split("### Response:")[-1].strip() if "### Response:" in full_text else full_text.strip()

    # Regex extraction for the three mandatory blocks
    tags_match = re.search(r'<tags>(.*?)</tags>', actual_gen, re.DOTALL)
    think_match = re.search(r'<think>(.*?)</think>', actual_gen, re.DOTALL)
    code_match = re.search(r'<code>(.*?)</code>', actual_gen, re.DOTALL)

    think_text = think_match.group(1).strip() if think_match else ""
    code_text = code_match.group(1).strip() if code_match else ""

    # FAR requires all three tags to be present and closed correctly
    has_all_tags = all([tags_match, think_match, code_match])

    # Calculate density of the thinking process
    total_len = len(actual_gen)
    density = (len(think_text) / total_len) if total_len > 0 else 0

    return {
        "adherent": has_all_tags,
        "density": density,
        "code_text": code_text
    }

def run_python_sandbox(code, input_data, timeout=5):
    """Executes the generated code against input in a subprocess."""
    with tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w') as tmp:
        tmp.write(code)
        tmp_path = tmp.name
    try:
        result = subprocess.run(
            [sys.executable, tmp_path],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=timeout
        )
        return result.stdout.strip(), result.stderr
    except subprocess.TimeoutExpired:
        return None, "TIMEOUT"
    except Exception as e:
        return None, str(e)
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

def evaluate_final():
    # 1. Load Ground Truth (Indexing by ID for O(1) matching)
    with open(GROUND_TRUTH_FILE, 'r') as f:
        truth = {str(json.loads(line)['id']): json.loads(line) for line in f}

    print(f"✅ Loaded {len(truth)} ground truth test cases.")

    stats = {
        "EASY": {"pass": 0, "total": 0, "far": 0, "density": []},
        "MEDIUM": {"pass": 0, "total": 0, "far": 0, "density": []},
        "HARD": {"pass": 0, "total": 0, "far": 0, "density": []}
    }

    # 2. Process Matched Solutions
    with open(MODEL_SOLUTIONS_FILE, 'r') as f:
        for line in tqdm(f, desc="⚖️ Running Final Judge"):
            sol = json.loads(line)
            task_id = str(sol.get('id'))

            if task_id not in truth:
                continue

            ref = truth[task_id]
            diff = ref.get('difficulty', 'EASY').upper()
            parsed = parse_stage2_response(sol['prediction'])

            # --- METRIC 1: FAR ---
            if parsed['adherent']:
                stats[diff]["far"] += 1

            # --- METRIC 2: Think Density ---
            stats[diff]["density"].append(parsed['density'])

            # --- METRIC 3: Pass@1 (Execution) ---
            if parsed['code_text']:
                passed_all = True
                test_cases = ref.get('test_cases', [])
                for test in test_cases:
                    actual, err = run_python_sandbox(parsed['code_text'], test['input'])
                    if actual is None or actual.strip() != test['output'].strip():
                        passed_all = False
                        break

                if passed_all and test_cases:
                    stats[diff]["pass"] += 1

            stats[diff]["total"] += 1

    # 3. Final Report Generation
    print("\n" + "="*70)
    print(f"🚀 FINAL STAGE 2 SCOREBOARD (ID-MATCHED)")
    print("="*70)
    print(f"{'DIFFICULTY':<12} | {'PASS@1':<12} | {'FAR':<12} | {'THINK DENSITY'}")
    print("-" * 70)

    total_p, total_t = 0, 0
    for d in ["EASY", "MEDIUM", "HARD"]:
        s = stats[d]
        if s['total'] == 0: continue

        total_p += s['pass']
        total_t += s['total']

        pass_rate = (s['pass'] / s['total']) * 100
        far_rate = (s['far'] / s['total']) * 100
        avg_density = (sum(s['density']) / len(s['density'])) * 100 if s['density'] else 0

        print(f"{d:<12} | {pass_rate:>7.1f}%     | {far_rate:>7.1f}%     | {avg_density:>7.1f}%")

    overall_pass = (total_p / total_t * 100) if total_t > 0 else 0
    print("-" * 70)
    print(f"{'OVERALL':<12} | {overall_pass:>7.1f}%     | {'-'*12} | {'-'*12}")
    print("="*70)

if __name__ == "__main__":
    evaluate_final()

✅ Loaded 200 ground truth test cases.


⚖️ Running Final Judge: 0it [00:00, ?it/s]


🚀 FINAL STAGE 2 SCOREBOARD (ID-MATCHED)
DIFFICULTY   | PASS@1       | FAR          | THINK DENSITY
----------------------------------------------------------------------
EASY         |     8.3%     |    95.0%     |    57.9%
MEDIUM       |     7.5%     |    91.2%     |    55.0%
HARD         |     8.3%     |    96.7%     |    56.8%
----------------------------------------------------------------------
OVERALL      |     8.0%     | ------------ | ------------


In [6]:
import json
import subprocess
import sys
import tempfile
import os
import re
from tqdm.auto import tqdm

# --- CONFIG ---
MODEL_SOLUTIONS_FILE = "/content/drive/MyDrive/SLM_DATA/stage2_FINAL_EVAL_MATCHED.jsonl"
GROUND_TRUTH_FILE = "/content/drive/MyDrive/SLM_DATA/test_suite_200_FINAL.jsonl"

def parse_code(full_text):
    actual_gen = full_text.split("### Response:")[-1].strip() if "### Response:" in full_text else full_text.strip()
    code_match = re.search(r'<code>(.*?)</code>', actual_gen, re.DOTALL)
    return code_match.group(1).strip() if code_match else None

def run_sandbox_with_logs(code, input_data, timeout=3):
    with tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w') as tmp:
        tmp.write(code)
        tmp_path = tmp.name
    try:
        result = subprocess.run(
            [sys.executable, tmp_path],
            input=input_data, capture_output=True, text=True, timeout=timeout
        )
        return result.stdout.strip(), result.stderr, "SUCCESS"
    except subprocess.TimeoutExpired:
        return None, None, "TLE"
    except Exception as e:
        return None, str(e), "RE"
    finally:
        if os.path.exists(tmp_path): os.remove(tmp_path)

def failure_analysis():
    with open(GROUND_TRUTH_FILE, 'r') as f:
        truth = {str(json.loads(line)['id']): json.loads(line) for line in f}

    error_counts = {"WA": 0, "TLE": 0, "RE": 0, "NO_CODE": 0}

    with open(MODEL_SOLUTIONS_FILE, 'r') as f:
        for line in tqdm(f, desc="🔍 Analyzing Failures"):
            sol = json.loads(line)
            ref = truth.get(str(sol.get('id')))
            if not ref: continue

            code = parse_code(sol['prediction'])
            if not code:
                error_counts["NO_CODE"] += 1
                continue

            passed_all = True
            current_error = "NONE"

            for test in ref.get('test_cases', []):
                actual, err, status = run_sandbox_with_logs(code, test['input'])

                if status == "TLE":
                    current_error = "TLE"
                    passed_all = False
                    break
                elif status == "RE" or (err and len(err) > 0):
                    current_error = "RE"
                    passed_all = False
                    break
                elif actual != test['output'].strip():
                    current_error = "WA"
                    passed_all = False
                    break

            if not passed_all:
                error_counts[current_error] += 1

    print("\n" + "="*40)
    print("🔥 FAILURE HEATMAP ANALYSIS")
    print("-" * 40)
    total_fails = sum(error_counts.values())
    for err, count in error_counts.items():
        pct = (count / total_fails * 100) if total_fails > 0 else 0
        print(f"{err:<10}: {count:>3} cases ({pct:>5.1f}%)")
    print("="*40)

failure_analysis()

🔍 Analyzing Failures: 0it [00:00, ?it/s]


🔥 FAILURE HEATMAP ANALYSIS
----------------------------------------
WA        : 127 cases ( 69.0%)
TLE       :   6 cases (  3.3%)
RE        :  46 cases ( 25.0%)
NO_CODE   :   5 cases (  2.7%)


In [7]:
def failure_case_study(target_error="WA", limit=1):
    with open(GROUND_TRUTH_FILE, 'r') as f:
        truth = {str(json.loads(line)['id']): json.loads(line) for line in f}

    found = 0
    with open(MODEL_SOLUTIONS_FILE, 'r') as f:
        for line in f:
            sol = json.loads(line)
            ref = truth.get(str(sol.get('id')))
            code = parse_code(sol['prediction'])

            if not code or not ref: continue

            # Check the first test case to identify the error type
            actual, err, status = run_sandbox_with_logs(code, ref['test_cases'][0]['input'])

            current_status = "SUCCESS"
            if status == "TLE": current_status = "TLE"
            elif status == "RE" or (err and len(err) > 0): current_status = "RE"
            elif actual != ref['test_cases'][0]['output'].strip(): current_status = "WA"

            if current_status == target_error:
                print(f"\n🕵️ CASE STUDY: {target_error}")
                print(f"Problem ID: {sol.get('id')}")
                print(f"Difficulty: {ref.get('difficulty')}")
                print("-" * 30)
                # Print the THINK block to see the logic
                think_content = re.search(r'<think>(.*?)</think>', sol['prediction'], re.DOTALL)
                print("🧠 MODEL'S INTERNAL REASONING:")
                print(think_content.group(1).strip() if think_content else "No think block found.")
                print("-" * 30)
                print(f"Expected: {repr(ref['test_cases'][0]['output'].strip())}")
                print(f"Actual  : {repr(actual)}")
                if err: print(f"Error Log: {err}")

                found += 1
                if found >= limit: break

# Run one of each
failure_case_study("WA")
failure_case_study("RE")


🕵️ CASE STUDY: WA
Problem ID: vfc_113116
Difficulty: EASY
------------------------------
🧠 MODEL'S INTERNAL REASONING:
```
[DIFFICULTY]: MEDIUM
[TOPICS]: MATH, GREEDY, DYNAMIC_PROGRAMMING
[GOAL]: Determine if a valid sequence of game winners exists given the number of players, wins for player 1, and wins for player 2.
[LOGIC]: The problem reduces to solving a system of linear equations for the number of games played by each player, ensuring non-negative integer solutions.
[PSEUDOCODE]:
```python
def solve():
    n, x, y = map(int, input().split())
    total_games = n - 1
    if x + y != total_games:
        print(-1)
        return

    # Initialize game counts
    game_counts = [0] * (n + 1)
    game_counts[1] = x
    game_counts[2] = y

    for i in range(3, n + 1):
        game_counts[i] = game_counts[i - 1] + game_counts[i - 2]

    if game_counts[n] != 0:
        print(-1)
    else:
        print(game_counts[1], game_counts[2], end=' ')
        print()

solve()
```
[CORNER CASES]